In [5]:
import os
import random
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import torch
import torch.nn as nn
from torchvision import models, transforms
import matplotlib.pyplot as plt
import cv2

# Configuration
SEED = 42
IMG_SIZE = 224
MODEL_PATH = "outputs/models/best_densenet121_tb.pth"
SHAP_VALUES_PATH = "outputs/shap_balanced/shap_values.npy"
SHAP_METADATA_PATH = "outputs/shap_balanced/shap_metadata.csv"
OUTPUT_DIR = "outputs/faithfulness"
PLOT_DIR = os.path.join(OUTPUT_DIR, "plots")
PANEL_DIR = os.path.join(OUTPUT_DIR, "example_occlusion_panels")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)
os.makedirs(PANEL_DIR, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

MASK_PERCENTAGES = [5, 10, 15, 20, 30]

# Reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_seed(SEED)

Using device: cuda


In [6]:
# Transform
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Load DenseNet121
model = models.densenet121(weights=None)
num_features = model.classifier.in_features
model.classifier = nn.Linear(num_features, 2)

model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model = model.to(DEVICE)
model.eval()

# Helper Functions
def load_image_tensor(image_path):
    image = Image.open(image_path).convert("RGB")
    tensor = transform(image)
    return tensor
    
def denormalize_tensor(tensor):
    image = tensor.detach().cpu().numpy().transpose(1, 2, 0)
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    image = std * image + mean
    image = np.clip(image, 0, 1)

    return image

def normalize_tensor_from_image(image_np):
    """
    Convert display image in range [0,1] back to normalized tensor.
    """
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    image_np = np.clip(image_np, 0, 1)
    normalized = (image_np - mean) / std
    tensor = torch.tensor(normalized.transpose(2, 0, 1), dtype=torch.float32)
    return tensor

def get_tb_probability(model, tensor):
    model.eval()
    tensor = tensor.unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        logits = model(tensor)
        probs = torch.softmax(logits, dim=1)
        tb_prob = probs[0, 1].item()
        prediction = torch.argmax(probs, dim=1).item()
    return tb_prob, prediction

def shap_to_heatmap(shap_value):
    """
    Convert SHAP value to 2D heatmap.
    Expected shape: C x H x W
    """
    shap_value = np.asarray(shap_value)

    if shap_value.ndim != 3:
        raise ValueError(f"Expected 3D SHAP value, got {shap_value.shape}")
    if shap_value.shape[0] == 3:
        heatmap = np.abs(shap_value).mean(axis=0)
    elif shap_value.shape[-1] == 3:
        heatmap = np.abs(shap_value).mean(axis=-1)
    else:
        raise ValueError(f"Unexpected SHAP shape: {shap_value.shape}")
    heatmap = heatmap - heatmap.min()
    heatmap = heatmap / (heatmap.max() + 1e-8)
    return heatmap

def create_mask_from_heatmap(heatmap, percentage, mode="high"):
    """
    Create binary mask from SHAP heatmap.

    mode:
    - high: mask top SHAP pixels
    - low: mask lowest SHAP pixels
    - random: mask random pixels
    """
    flat = heatmap.flatten()
    n_pixels = len(flat)
    n_mask = int((percentage / 100.0) * n_pixels)

    mask_flat = np.zeros(n_pixels, dtype=np.uint8)

    if mode == "high":
        indices = np.argsort(flat)[-n_mask:]
    elif mode == "low":
        indices = np.argsort(flat)[:n_mask]
    elif mode == "random":
        indices = np.random.choice(n_pixels, size=n_mask, replace=False)
    else:
        raise ValueError(f"Unknown mode: {mode}")
    mask_flat[indices] = 1
    mask = mask_flat.reshape(heatmap.shape)
    return mask

def apply_mask(image_np, mask, fill_value="mean"):
    """
    Apply occlusion mask to image.

    image_np shape: H x W x 3, range [0,1]
    mask shape: H x W, values 0 or 1
    """
    masked_image = image_np.copy()

    if fill_value == "mean":
        replacement = image_np.mean(axis=(0, 1))
    elif fill_value == "black":
        replacement = np.array([0.0, 0.0, 0.0])
    elif fill_value == "gray":
        replacement = np.array([0.5, 0.5, 0.5])
    else:
        raise ValueError(f"Unknown fill value: {fill_value}")
    masked_image[mask == 1] = replacement
    return masked_image

def save_occlusion_panel(original, high_masked, low_masked, random_masked, save_path, title):
    plt.figure(figsize=(16, 4))

    plt.subplot(1, 4, 1)
    plt.imshow(original)
    plt.axis("off")
    plt.title("Original")

    plt.subplot(1, 4, 2)
    plt.imshow(high_masked)
    plt.axis("off")
    plt.title("High-SHAP masked")

    plt.subplot(1, 4, 3)
    plt.imshow(low_masked)
    plt.axis("off")
    plt.title("Low-SHAP masked")

    plt.subplot(1, 4, 4)
    plt.imshow(random_masked)
    plt.axis("off")
    plt.title("Random masked")

    plt.suptitle(title)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()

In [7]:
# Load SHAP Outputs
shap_values = np.load(SHAP_VALUES_PATH)
meta_df = pd.read_csv(SHAP_METADATA_PATH)
print("SHAP values shape:", shap_values.shape)
print("Metadata shape:", meta_df.shape)

# Faithfulness Occlusion Experiment
results = []
for idx in tqdm(range(len(meta_df)), desc="Running faithfulness tests"):
    row = meta_df.iloc[idx]
    image_id = row["image_id"]
    image_path = row["image_path"]
    true_label = int(row["true_label"])

    original_tensor = load_image_tensor(image_path)
    original_image = denormalize_tensor(original_tensor)
    original_prob, original_pred = get_tb_probability(model, original_tensor)
    shap_heatmap = shap_to_heatmap(shap_values[idx])
    for percentage in MASK_PERCENTAGES:
        for mode in ["high", "low", "random"]:
            mask = create_mask_from_heatmap(
                shap_heatmap,
                percentage=percentage,
                mode=mode
            )
            masked_image = apply_mask(
                original_image,
                mask,
                fill_value="mean"
            )
            masked_tensor = normalize_tensor_from_image(masked_image)
            masked_prob, masked_pred = get_tb_probability(model, masked_tensor)
            probability_drop = original_prob - masked_prob
            absolute_probability_change = abs(original_prob - masked_prob)

            results.append({
                "image_id": image_id,
                "image_path": image_path,
                "true_label": true_label,
                "original_prediction": original_pred,
                "original_tb_probability": original_prob,
                "mask_type": mode,
                "mask_percentage": percentage,
                "masked_prediction": masked_pred,
                "masked_tb_probability": masked_prob,
                "probability_drop": probability_drop,
                "absolute_probability_change": absolute_probability_change
            })
results_df = pd.DataFrame(results)
results_path = os.path.join(OUTPUT_DIR, "faithfulness_results.csv")
results_df.to_csv(results_path, index=False)
print(f"Saved detailed results to {results_path}")

# Summary Statistics
summary_df = results_df.groupby(
    ["mask_type", "mask_percentage"]
).agg(
    mean_probability_drop=("probability_drop", "mean"),
    std_probability_drop=("probability_drop", "std"),
    mean_absolute_probability_change=("absolute_probability_change", "mean"),
    std_absolute_probability_change=("absolute_probability_change", "std"),
    n=("image_id", "count")
).reset_index()
summary_path = os.path.join(OUTPUT_DIR, "faithfulness_summary.csv")
summary_df.to_csv(summary_path, index=False)
print("\nFaithfulness summary:")
print(summary_df)

SHAP values shape: (45, 3, 224, 224)
Metadata shape: (45, 7)


Running faithfulness tests: 100%|███████████████| 45/45 [00:09<00:00,  4.87it/s]

Saved detailed results to outputs/faithfulness/faithfulness_results.csv

Faithfulness summary:
   mask_type  mask_percentage  mean_probability_drop  std_probability_drop  \
0       high                5              -0.443669              0.361142   
1       high               10              -0.415489              0.370953   
2       high               15              -0.366047              0.378072   
3       high               20              -0.324263              0.398098   
4       high               30              -0.253148              0.420242   
5        low                5              -0.016496              0.210654   
6        low               10              -0.136930              0.269391   
7        low               15              -0.223497              0.316164   
8        low               20              -0.266998              0.325002   
9        low               30              -0.355575              0.357877   
10    random                5              -0.2

In [ ]:
# Plot 1: Mean Probability Drop
plt.figure(figsize=(8, 5))
for mode in ["high", "low", "random"]:
    subset = summary_df[summary_df["mask_type"] == mode]

    plt.plot(
        subset["mask_percentage"],
        subset["mean_probability_drop"],
        marker = "o",
        label = mode
    )

plt.xlabel("Masked Area (%)")
plt.ylabel("Mean TB Probability Drop")
plt.title("Faithfulness Test: Probability Drop After Occlusion")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "probability_drop_lineplot.png"), dpi=300)
plt.close()

# Plot 2: Boxplot at 20% Masking
boxplot_percentage = 20
box_df = results_df[results_df["mask_percentage"] == boxplot_percentage]
plt.figure(figsize=(7, 5))

data_to_plot = [
    box_df[box_df["mask_type"] == "high"]["probability_drop"],
    box_df[box_df["mask_type"] == "low"]["probability_drop"],
    box_df[box_df["mask_type"] == "random"]["probability_drop"]
]
plt.boxplot(data_to_plot, tick_labels=["High SHAP", "Low SHAP", "Random"])
plt.ylabel("TB Probability Drop")
plt.title(f"Probability Drop Distribution at {boxplot_percentage}% Masking")
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "probability_drop_boxplot_20_percent.png"), dpi=300)
plt.close()

# Save Example Occlusion Panels
examples_saved = 0
max_examples = 5
example_percentage = 20

for idx in range(len(meta_df)):
    row = meta_df.iloc[idx]
    image_id = row["image_id"]
    image_path = row["image_path"]

    original_tensor = load_image_tensor(image_path)
    original_image = denormalize_tensor(original_tensor)
    shap_heatmap = shap_to_heatmap(shap_values[idx])

    high_mask = create_mask_from_heatmap(shap_heatmap, example_percentage, mode="high")
    low_mask = create_mask_from_heatmap(shap_heatmap, example_percentage, mode="low")
    random_mask = create_mask_from_heatmap(shap_heatmap, example_percentage, mode="random")

    high_masked = apply_mask(original_image, high_mask, fill_value="mean")
    low_masked = apply_mask(original_image, low_mask, fill_value="mean")
    random_masked = apply_mask(original_image, random_mask, fill_value="mean")
    title = f"Occlusion Examples | ID: {image_id} | Masked area: {example_percentage}%"
    save_path = os.path.join(
        PANEL_DIR,
        f"{image_id}_occlusion_panel.png"
    )
    save_occlusion_panel(
        original_image,
        high_masked,
        low_masked,
        random_masked,
        save_path,
        title
    )
    examples_saved += 1
    if examples_saved >= max_examples:
        break
print(f"Saved {examples_saved} occlusion example panels.")

In [6]:
import os
import pandas as pd
from scipy.stats import wilcoxon

RESULTS_PATH = "outputs/faithfulness/faithfulness_results.csv"
OUTPUT_DIR = "outputs/faithfulness"
results_df = pd.read_csv(RESULTS_PATH)
test_rows = []

for percentage in sorted(results_df["mask_percentage"].unique()):
    subset = results_df[results_df["mask_percentage"] == percentage]
    high = subset[subset["mask_type"] == "high"].sort_values("image_id")["probability_drop"].values
    low = subset[subset["mask_type"] == "low"].sort_values("image_id")["probability_drop"].values
    random = subset[subset["mask_type"] == "random"].sort_values("image_id")["probability_drop"].values

    if len(high) == len(low) and len(high) > 0:
        stat, p = wilcoxon(high, low)

        test_rows.append({
            "mask_percentage": percentage,
            "comparison": "high_vs_low",
            "wilcoxon_statistic": stat,
            "p_value": p
        })
    if len(high) == len(random) and len(high) > 0:
        stat, p = wilcoxon(high, random)

        test_rows.append({
            "mask_percentage": percentage,
            "comparison": "high_vs_random",
            "wilcoxon_statistic": stat,
            "p_value": p
        })
stats_df = pd.DataFrame(test_rows)
stats_path = os.path.join(OUTPUT_DIR, "faithfulness_wilcoxon_tests.csv")
stats_df.to_csv(stats_path, index=False)
print(stats_df)
print(f"Saved statistical tests to {stats_path}")

   mask_percentage      comparison  wilcoxon_statistic       p_value
0                5     high_vs_low                30.0  3.701643e-09
1                5  high_vs_random                71.0  5.841630e-07
2               10     high_vs_low               102.0  9.597461e-06
3               10  high_vs_random                 5.0  1.818989e-11
4               15     high_vs_low               189.0  2.392230e-03
5               15  high_vs_random                 3.0  9.094947e-12
6               20     high_vs_low               385.0  7.446044e-01
7               20  high_vs_random                 0.0  1.818989e-12
8               30     high_vs_low               101.0  8.847996e-06
9               30  high_vs_random                 0.0  1.818989e-12
Saved statistical tests to outputs/faithfulness/faithfulness_wilcoxon_tests.csv
